# Demo 12 - Failed-logon spike detection (rolling baseline)

**Fast** (hourly series) · **Pool:** Small · **Visual:** time series with control band

**The question:** is this hour's failed-sign-in count unusual **for us**, right now?

A fixed threshold is wrong in both directions at once: too noisy for a large tenant, too
quiet for a small one. Instead we compare each hour against the average and spread of the
preceding day, so the bar moves with your actual traffic and a quiet weekend does not need
its own rule.

Rolling windows, standard deviations and a plotted control band are natural in pandas and
clumsy in KQL.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

- `LOOKBACK_DAYS` - how much history to plot.
- `WINDOW` - how many hours of recent history the baseline is built from. 24 means "compare
  each hour against the last day".
- `Z` - how many standard deviations from that baseline counts as a spike. Three is the
  conventional choice.

In [ ]:
WORKSPACE = "your-workspace-name"
LOOKBACK_DAYS = 14
WINDOW = 24     # rolling window in hours
Z = 3.0

## 3. Count failed sign-ins per hour

Same success-code logic as the other identity notebooks: anything that is not one of the
known success codes is a failure, and a null error code counts as a failure too rather than
being silently dropped.

`asfreq("h")` fills in hours that had no failures with a zero. That matters more than it
sounds - a missing hour would shift the rolling window along and quietly corrupt every
calculation after it.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pyspark.sql.types import StructType, StructField, StringType

status = StructType([StructField("errorCode", StringType(), True)])
ok = ["0","50125","50140","70043","70044"]

df = data_provider.read_table("SigninLogs", WORKSPACE)
fails = (df.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
           .withColumn("ec", F.from_json("Status", status).getField("errorCode"))
           # a null errorCode is not a success - treat it as a failure, matching demo02/demo04
           .filter(F.col("ec").isNull() | ~F.col("ec").isin(ok))
           .groupBy(F.date_trunc("hour","TimeGenerated").alias("hour")).agg(F.count("*").alias("fails"))
           .orderBy("hour")).toPandas()

if fails.empty:
    s = pd.Series(dtype="float64")
else:
    idx = pd.to_datetime(fails["hour"])
    try:
        s = fails.set_index(idx)["fails"].asfreq("h").fillna(0)   # pandas >= 2.2
    except ValueError:
        s = fails.set_index(idx)["fails"].asfreq("H").fillna(0)   # pandas < 2.2
print("hours:", len(s))

## 4. Compare each hour against its own recent baseline

A fixed threshold ("alert above 100 failures") is wrong in both directions at once: too
noisy for a large tenant, too quiet for a small one, and wrong for either at 3am.

A **rolling z-score** adapts instead. For each hour, take the mean and standard deviation of
the previous `WINDOW` hours, then ask how many standard deviations above that mean the
current hour sits. That number is the z-score.

- z = 0: exactly average for the recent period
- z = 3: three standard deviations high, which is genuinely unusual
- The threshold moves with your traffic, so a quiet weekend does not need its own rule

The green line is the rolling mean, the shaded band is plus or minus three sigma, and the
red dots are hours that broke out of it.

**What to look for:** a single red dot is usually one noisy service account. A run of them
climbing out of the band is the shape of a spray or brute-force campaign.

In [ ]:
if s.empty:
    print(f"No failed sign-ins in the last {LOOKBACK_DAYS} days - raise LOOKBACK_DAYS.")
else:
    mean = s.rolling(WINDOW, min_periods=6).mean()
    std  = s.rolling(WINDOW, min_periods=6).std().fillna(0)
    z = (s - mean) / std.replace(0, np.nan)
    spikes = s[z > Z]

    plt.figure(figsize=(14,5))
    plt.plot(s.index, s.values, color="#2c3e50", label="failed sign-ins/hr")
    plt.plot(mean.index, mean.values, color="#27ae60", label=f"{WINDOW}h rolling mean")
    plt.fill_between(mean.index, mean - Z*std, mean + Z*std, color="#27ae60", alpha=.15,
                     label=f"+/-{int(Z)} sigma")
    plt.scatter(spikes.index, spikes.values, color="red", zorder=5, label="spike (z>3)")
    plt.title("Failed sign-ins per hour with rolling control band")
    plt.ylabel("failures/hr"); plt.legend(); plt.tight_layout(); plt.show()
    print("spike hours:", len(spikes))

## Why a notebook beats KQL here

A rolling z-score adapts the threshold to the recent baseline instead of a fixed number, so it catches relative spikes and ignores steady noise. Expressing rolling windows, sigma bands and the plot is trivial in pandas/matplotlib and clumsy in KQL.